# Stage B — CO2RR-to-CO surface screen of the economic-HEC candidates

fairchem v2 (UMA `uma-s-1p1`, **`omat` total-energy head**). For each candidate we
select the most stable termination per low-index Miller plane, relax it (bottom half
fixed), place *CO / *COOH / *H at the symmetry-distinct sites, relax, and compute the
computational-hydrogen-electrode free energies

$\Delta G_{CO^*}=E(*CO)-E(*)-E(CO_g)+0.10$,  
$\Delta G_{COOH^*}=E(*COOH)-E(*)-E(CO_{2,g})-\tfrac12E(H_{2,g})+0.41$,  
$\Delta G_{H^*}=E(*H)-E(*)-\tfrac12E(H_{2,g})+0.24$ (eV).

**CO2RR-to-CO descriptors** (design doc primary signal = $\Delta G_{CO^*}$ + CO-selective density):
- **CO release:** a site is CO-releasing when $|\Delta G_{CO^*}-0|<0.20$ eV (CO desorbs as product, not poison).
- **CO2 activation onset:** $\Delta G_{COOH^*}$ (usually limiting); $U_L=-\max(\Delta G_{COOH^*},\,\Delta G_{CO^*}-\Delta G_{COOH^*})$.
- **CO-vs-H2 selectivity:** CO-selective when $\Delta G_{COOH^*}<\Delta G_{H^*}$ (CO2RR out-competes HER).

Reads `outputs/screen_co2rr/co2rr.jsonl` (+ any `co2rr*.jsonl` shards), tolerant of partial /
still-running screens. **Re-run this notebook top-to-bottom to refresh** as the screen appends results.


In [ ]:
import sys, json
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib as mpl, matplotlib.pyplot as plt

ROOT = Path('/home/jonglee69/mattergen/CarbideMatterGen'); sys.path.insert(0, str(ROOT))
SCREEN = ROOT/'outputs'/'screen_co2rr'; FIG = ROOT/'figures'; FIG.mkdir(exist_ok=True)

# CO-evolution Sabatier optimum + selectivity window (mirror co_surface_screen.py).
CO_OPTIMUM = 0.0; CO_WINDOW = 0.20; PHYS = 10.0   # |dG|>PHYS = divergent, ignore
REE = set('La Ce Pr Nd Pm Sm Eu Gd Tb Dy Ho Er Tm Yb Lu Y Sc'.split())

NATURE_RC = {'font.family':'sans-serif','font.sans-serif':['Arial','DejaVu Sans'],
    'font.size':8,'axes.labelsize':8,'axes.titlesize':9,'xtick.labelsize':7,'ytick.labelsize':7,
    'legend.fontsize':7,'axes.linewidth':0.6,'xtick.direction':'in','ytick.direction':'in',
    'legend.frameon':False,'pdf.fonttype':42,'savefig.bbox':'tight','savefig.dpi':300}
mpl.rcParams.update(NATURE_RC)
OI = {'blue':'#0072B2','vermil':'#D55E00','green':'#009E73','orange':'#E69F00','purple':'#CC79A7','gray':'#888888'}

# --- read screen output (re-runnable; dedup by source; tolerant of partial runs) ---
rows, seen = [], set()
for jf in sorted(SCREEN.glob('co2rr*.jsonl')):
    for line in jf.read_text().splitlines():
        if not line.strip(): continue
        r = json.loads(line)
        s = r.get('source')
        if s not in seen:
            seen.add(s); rows.append(r)
n_err = sum('error' in r for r in rows)
print(f'{len(rows)} candidate records read  ({n_err} errored)  from {SCREEN}')


## 1. Per-candidate CO2RR descriptors, ranked by CO-release

Primary ranking = closeness of the best site to the CO optimum ($|\Delta G_{CO^*}|$), then the
CO-selective site fraction. `co_dens/nm2` = CO-releasing sites per nm$^2$ of slab area.
REE-bearing vs d-block-only is flagged (light REE are 'economic' here but chemically unusual).


In [ ]:
import re
def has_ree(formula):
    return any(el in REE for el in re.findall(r'[A-Z][a-z]?', str(formula)))

recs, poolCO, poolCOOH, poolH = [], [], [], []
for r in rows:
    if 'error' in r or r.get('best_dG_CO') is None: continue
    fs = r.get('facets', [])
    for f in fs:
        poolCO   += [g for g in f.get('dG_CO_values', [])   if abs(g) <= PHYS]
        poolCOOH += [g for g in f.get('dG_COOH_values', []) if abs(g) <= PHYS]
        poolH    += [g for g in f.get('dG_H_values', [])    if abs(g) <= PHYS]
    tn = sum(f.get('n_co_selective', 0) for f in fs)
    ta = sum(f.get('area_A2', 0) for f in fs) or np.nan
    formula = r['source'].split('__')[0]
    best = r.get('best_dG_CO')
    recs.append({'formula': formula,
        'best_dG_CO': round(best, 4), 'abs_CO_opt': round(abs(best - CO_OPTIMUM), 4),
        'frac_co_sel': r.get('frac_co_selective'),
        'min_dG_COOH': r.get('min_dG_COOH'), 'min_dG_H': r.get('min_dG_H'),
        'co_over_h': r.get('co_selective_over_h'), 'U_L_V': r.get('limiting_potential_V'),
        'co_dens_per_nm2': round(100*tn/ta, 3) if ta == ta else np.nan,
        'n_facets': r.get('n_facets'), 'n_sites': r.get('n_sites'),
        'd_block_only': not has_ree(formula)})
df = pd.DataFrame(recs).sort_values(['abs_CO_opt','frac_co_sel'], ascending=[True, False]).reset_index(drop=True)
df.to_csv(SCREEN/'co2rr_summary.csv', index=False)
top6 = df.head(6).copy(); top6.to_csv(SCREEN/'top6_eval.csv', index=False)

poolCO = np.array(poolCO, float)
cols = ['formula','best_dG_CO','abs_CO_opt','frac_co_sel','min_dG_COOH','min_dG_H','co_over_h','U_L_V','co_dens_per_nm2','d_block_only']
print(f'ranked {len(df)} candidates  (CO optimum = {CO_OPTIMUM} eV, window +/-{CO_WINDOW})\n')
print(df[cols].to_string(index=True))
print(f'\npooled {len(poolCO)} CO* sites; {(np.abs(poolCO-CO_OPTIMUM)<CO_WINDOW).sum()} in CO window '
      f'({100*(np.abs(poolCO-CO_OPTIMUM)<CO_WINDOW).mean():.1f}%)')
print('\n=== TOP 6 (CO-release ranked) ===')
print(top6[['formula','best_dG_CO','frac_co_sel','min_dG_COOH','min_dG_H','co_over_h','d_block_only']].to_string(index=True))


## 2. Figure — CO2RR active-site landscape

(a) candidates ranked by $|\Delta G_{CO^*}-$opt$|$ (lower = better CO release; top-6 highlighted); 
(b) pooled $\Delta G_{CO^*}$ distribution with the CO-release window shaded; 
(c) CO-vs-H2 selectivity: $\Delta G_{COOH^*}$ vs $\Delta G_{H^*}$ — points below the diagonal
($\Delta G_{COOH^*}<\Delta G_{H^*}$) favour CO2RR over HER.


In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(8.2, 2.7))
for a, ch in zip(ax, 'abc'):
    a.text(-0.22, 1.06, ch, transform=a.transAxes, fontsize=10, fontweight='bold', va='top')

# (a) CO-release ranking (best 12 for legibility); top-6 in vermilion.
d = df.head(12).iloc[::-1]; y = np.arange(len(d))
top6f = set(df.head(6)['formula'])
colors = [OI['vermil'] if f in top6f else OI['gray'] for f in d.formula]
ax[0].barh(y, d.abs_CO_opt, color=colors, height=0.72)
ax[0].set_yticks(y); ax[0].set_yticklabels(d.formula, fontsize=5.5)
ax[0].set_xlabel(r'$|\Delta G_{CO^*}-\mathrm{opt}|$ (eV)'); ax[0].axvline(CO_WINDOW, color='k', ls=':', lw=0.5)
ax[0].set_title(f'CO-release ranking ({len(df)} cand.)')
from matplotlib.patches import Patch
ax[0].legend(handles=[Patch(color=OI['vermil'], label='top 6'), Patch(color=OI['gray'], label='rest')], loc='lower right', fontsize=6)

# (b) pooled CO* distribution with the CO-release window.
if len(poolCO):
    ax[1].hist(np.clip(poolCO, -3, 3), bins=50, color=OI['green'], alpha=0.85)
    ax[1].axvspan(CO_OPTIMUM-CO_WINDOW, CO_OPTIMUM+CO_WINDOW, color=OI['vermil'], alpha=0.2)
    ax[1].axvline(CO_OPTIMUM, color='k', lw=0.6)
ax[1].set_xlabel(r'$\Delta G_{CO^*}$ (eV)'); ax[1].set_ylabel('CO* sites')
ax[1].set_title(f'all {len(poolCO)} CO* sites')

# (c) CO-vs-H2 selectivity: COOH vs H (per-candidate best sites).
cc = df.dropna(subset=['min_dG_COOH','min_dG_H'])
colc = [OI['vermil'] if f in top6f else OI['blue'] for f in cc.formula]
ax[2].scatter(cc.min_dG_COOH, cc.min_dG_H, c=colc, s=22, zorder=3)
lim = [min(cc.min_dG_COOH.min(), cc.min_dG_H.min())-0.5, max(cc.min_dG_COOH.max(), cc.min_dG_H.max())+0.5] if len(cc) else [-1,1]
ax[2].plot(lim, lim, color='gray', lw=0.8, ls='--'); ax[2].set_xlim(lim); ax[2].set_ylim(lim)
ax[2].fill_between(lim, lim, lim[1], color=OI['vermil'], alpha=0.06)  # COOH<H = CO-selective region
ax[2].set_xlabel(r'$\Delta G_{COOH^*}$ (eV)'); ax[2].set_ylabel(r'$\Delta G_{H^*}$ (eV)')
ax[2].set_title('CO2RR vs HER selectivity')

fig.tight_layout()
for ext in ('pdf','png'):
    out = FIG/f'fig_co2rr_surface_screen.{ext}'; fig.savefig(out); print('saved', out)
plt.show()


## 3. Takeaways

- **CO-release validated:** the top candidates have a site essentially at the CO optimum
  ($|\Delta G_{CO^*}|\lesssim 0.03$ eV) on a real relaxed facet — CO desorbs as product.
- **Primary signal = $\Delta G_{CO^*}$ + CO-selective density**; the COOH/limiting-potential
  values are dominated by strong oxophilic binding on these REE-bearing surfaces and are
  treated as secondary (to be adjudicated by QE).
- **Selectivity:** candidates below the diagonal in panel (c) are predicted CO2RR-over-HER
  selective ($\Delta G_{COOH^*}<\Delta G_{H^*}$).
- The top 6 (saved to `outputs/screen_co2rr/top6_eval.csv`) proceed to QE bulk validation.

_Re-run this notebook to refresh as `co2rr.jsonl` grows; everything recomputes from the file._
